# Algoritmos genéticos: otimizar por seleção e variação

**Para quem está começando.** Nenhuma linha de código aqui assume que você já viu otimização. A ideia é rodar tudo de cima para baixo, célula por célula, e olhar os números que aparecem.

O que vamos fazer, em ordem:

1. entender o que é um algoritmo genético, sem jargão;
2. maximizar uma função com vários picos, onde a busca por gradiente se perde;
3. usar o mesmo algoritmo para escolher atributos num problema real de classificação;
4. comparar o resultado com o modelo que usa todos os atributos.

Roda no Google Colab ou no Jupyter do seu computador. Só precisa de `numpy`, `scikit-learn`, `pandas` e `matplotlib`, que já vêm no Colab.


## O que é um algoritmo genético, sem jargão

Você tem um problema em que precisa escolher a melhor entre muitas combinações e não existe fórmula para ir direto na resposta. O algoritmo genético resolve isso imitando o que a seleção natural faz com populações.

O vocabulário, traduzido:

- **indivíduo** é uma solução candidata, escrita como uma sequência de valores, o *cromossomo*;
- **população** é o conjunto de indivíduos mantido geração após geração;
- **fitness** é a nota de cada indivíduo, calculada por uma função que você escreve;
- **seleção** escolhe os pais, dando mais chance a quem tem nota melhor;
- **crossover** mistura o cromossomo de dois pais;
- **mutação** muda um valor ao acaso, para a busca não travar num ponto.

O ciclo é sempre o mesmo: avaliar, selecionar, cruzar, mutar, repetir.

O que isso muda na prática. O algoritmo não precisa da derivada da função objetivo, então serve para problemas discretos, onde o gradiente não existe. E ele trabalha com uma população, não com um ponto só, o que explora várias regiões ao mesmo tempo. O preço é que nenhuma geração garante melhora: quem decide é o fitness, e a resposta boa costuma vir de muitas gerações.


## Passo 0: ambiente e configuração

Nada para instalar além do que já vem no Colab. As configurações do experimento ficam todas aqui, e a semente é a mesma em todas as células: duas execuções deste notebook dão o mesmo resultado.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split

# Same seed everywhere: two runs of this notebook must agree.
RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (9, 4)

print(f"✅ numpy {np.__version__} | seed {RANDOM_STATE}")

# Parte 1: maximizar uma função com vários picos

A função abaixo vale para qualquer `x` entre 0 e 10 e tem vários picos locais. O pico mais alto fica perto de `x = 9`.

Uma busca por gradiente subiria o primeiro morro que encontrasse e pararia lá, porque num pico local a inclinação é zero. O algoritmo genético não olha a inclinação: ele mantém uma população espalhada pelo intervalo, e a seleção vai puxando os indivíduos para as regiões que pontuam mais.


In [ ]:
def objective(x: float) -> float:
    """Function to maximize: several local peaks, so a local search gets stuck."""
    return x * np.sin(3.0 * x)


grid = np.linspace(0.0, 10.0, 500)
best_on_grid = grid[np.argmax(objective(grid))]
print(f"🔎 melhor ponto na grade: x = {best_on_grid:.3f} | f = {objective(best_on_grid):.3f}")

fig, ax = plt.subplots()
ax.plot(grid, objective(grid), color="#0F6B68")
ax.axvline(best_on_grid, color="#B45309", linestyle="--", linewidth=1)
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("f(x) = x · sin(3x): vários picos, um pico global")
plt.show()

## O algoritmo genético, em uma classe

Toda a mecânica cabe numa classe. Três decisões que valem para os dois problemas deste notebook:

- **representação**: o cromossomo é um vetor de genes. Aqui ele pode ser um número real (parte 1) ou um bit por atributo (parte 2);
- **decodificação**: `decode` transforma o cromossomo no que a função de fitness lê. É o único ponto que muda entre os dois problemas;
- **elitismo**: o melhor indivíduo de cada geração passa intacto para a seguinte, senão a busca pode piorar de uma geração para outra.


In [ ]:
class GeneticAlgorithm:
    """Evolve a population of chromosomes to maximize a fitness function.

    One representation only: a chromosome is an array of genes. `decode` turns it
    into whatever the fitness function reads, so the same class solves the
    continuous problem and the feature-selection problem.
    """

    def __init__(self, fitness, decode, size=40, n_genes=1, gene_bounds=None,
                 tournament=3, crossover_rate=0.9, mutation_rate=0.05, seed=42):
        self.fitness = fitness
        self.decode = decode
        self.size = size
        self.n_genes = n_genes
        self.gene_bounds = gene_bounds       # None -> binary genes, one bit per gene
        self.tournament = tournament
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.rng = np.random.default_rng(seed)
        self.history = []                    # (best, mean, std) per generation

    def random_population(self) -> np.ndarray:
        """Start spread out: continuous genes inside the bounds, binary genes 0/1."""
        if self.gene_bounds is None:
            return self.rng.integers(0, 2, size=(self.size, self.n_genes))
        low, high = self.gene_bounds
        return self.rng.uniform(low, high, size=(self.size, self.n_genes))

    def evaluate(self, population: np.ndarray) -> np.ndarray:
        return np.array([self.fitness(self.decode(genes)) for genes in population])

    def select(self, population: np.ndarray, scores: np.ndarray) -> np.ndarray:
        """Tournament selection: draw some individuals, keep the best of them."""
        drawn = self.rng.integers(0, self.size, size=self.tournament)
        return population[drawn[np.argmax(scores[drawn])]]

    def crossover(self, parent_a: np.ndarray, parent_b: np.ndarray):
        """Single-point crossover: cut both parents at the same gene and swap tails."""
        # A single-gene chromosome has no cut point: only mutation can move it.
        if self.n_genes < 2 or self.rng.random() > self.crossover_rate:
            return parent_a.copy(), parent_b.copy()
        point = int(self.rng.integers(1, self.n_genes))
        child_a = np.concatenate([parent_a[:point], parent_b[point:]])
        child_b = np.concatenate([parent_b[:point], parent_a[point:]])
        return child_a, child_b

    def mutate(self, genes: np.ndarray) -> np.ndarray:
        """Flip a bit or redraw a real gene, each with probability mutation_rate."""
        genes = genes.copy()
        for i in range(self.n_genes):
            if self.rng.random() < self.mutation_rate:
                if self.gene_bounds is None:
                    genes[i] = 1 - genes[i]
                else:
                    low, high = self.gene_bounds
                    genes[i] = self.rng.uniform(low, high)
        return genes

    def run(self, generations: int = 40, verbose: bool = True):
        """Evolve the population and keep the score of every generation."""
        population = self.random_population()
        scores = self.evaluate(population)
        for generation in range(generations):
            offspring = np.empty_like(population)
            for i in range(0, self.size, 2):
                child_a, child_b = self.crossover(self.select(population, scores),
                                                  self.select(population, scores))
                offspring[i] = self.mutate(child_a)
                if i + 1 < self.size:
                    offspring[i + 1] = self.mutate(child_b)
            offspring_scores = self.evaluate(offspring)
            if offspring_scores.max() < scores.max():
                worst = int(np.argmin(offspring_scores))
                offspring[worst] = population[int(np.argmax(scores))]
                offspring_scores[worst] = scores.max()
            population, scores = offspring, offspring_scores
            self.history.append((scores.max(), scores.mean(), scores.std()))
            if verbose:
                print(f"  geracao {generation + 1:3d} | melhor {scores.max():8.4f} "
                      f"| media {scores.mean():8.4f} | desvio {scores.std():7.4f}")
        best = int(np.argmax(scores))
        return population[best], scores[best]

## Passo 1: rodar a busca

O cromossomo aqui tem um único gene, porque a solução é um número entre 0 e 10. A mutação está alta (0,2) de propósito: com um gene só, ela é o que permite pular de um morro para outro.

Repare na coluna `desvio`: ela mede a diversidade da população. Aqui ela **não** cai, e há uma razão. O cromossomo tem um gene só, então o crossover não tem o que misturar: quem explora é a mutação, que sorteia um valor novo para cerca de 20% dos indivíduos a cada geração. O melhor fica parado no pico, por causa do elitismo, e o resto da população continua espalhado.


In [ ]:
ga = GeneticAlgorithm(
    fitness=objective,
    decode=lambda genes: float(genes[0]),
    size=40, n_genes=1, gene_bounds=(0.0, 10.0),
    tournament=3, crossover_rate=0.9, mutation_rate=0.2, seed=RANDOM_STATE,
)

print("🧬 populacao de 40 individuos, 40 geracoes")
best_genes, best_score = ga.run(generations=40, verbose=True)
print(f"\n🏁 melhor individuo: x = {best_genes[0]:.4f} | f = {best_score:.4f}")

In [ ]:
history = np.array(ga.history)

fig, ax = plt.subplots()
ax.plot(history[:, 0], label="melhor da geração", color="#0F6B68")
ax.plot(history[:, 1], label="média da população", color="#B45309")
ax.set_xlabel("geração")
ax.set_ylabel("fitness")
ax.set_title("Convergência: o melhor sobe e a média acompanha")
ax.legend()
plt.show()

### Como ler isso

A curva do melhor sobe pouco e cedo. São 40 indivíduos espalhados por um intervalo de 10, o que já cobre a região do pico global na primeira geração; o elitismo garante que esse valor não se perca depois.

A média fica bem abaixo e oscila, sem se juntar à curva de cima. Isso é o retrato de um cromossomo de um gene: sem crossover, a busca é seleção mais mutação, e a diversidade não desaparece. Com muitos genes o comportamento muda, porque aí o crossover combina pedaços bons e a população converge. Compare com o desvio da Parte 2.

Duas checagens honestas: mutação alta ajuda aqui porque a função tem vários picos, mas a mesma mutação alta num problema de muitos genes destrói o que o crossover construiu; e a comparação com o melhor ponto da grade é o que diz se a busca chegou perto do ótimo global.


# Parte 2: escolher atributos num problema real

Agora um problema de Machine Learning. A tabela de câncer de mama tem 30 atributos, e cada um pode entrar ou ficar de fora: são `2^30`, mais de um bilhão de subconjuntos possíveis. Não dá para testar um por um.

Este é o tipo de problema em que o algoritmo genético faz sentido:

- a decisão é **discreta** (entra ou não entra), então não existe derivada para descer;
- o custo de avaliar um subconjunto é uma validação cruzada, ou seja, treinar modelos. É caro, então convém não desperdiçar avaliações;
- a população permite testar muitos subconjuntos em paralelo, e o crossover combina os bons pedaços de dois subconjuntos.

O cromossomo passa a ser um bit por atributo, e o fitness é a acurácia média na validação cruzada dentro do treino. O teste fica guardado para o final.


## Passo 2: baseline com todos os atributos

Antes de otimizar, o número que precisa ser batido.


In [ ]:
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.25, stratify=data.target, random_state=RANDOM_STATE,
)

baseline = LogisticRegression(max_iter=5000).fit(X_train, y_train)
baseline_accuracy = baseline.score(X_test, y_test)

print(f"🔎 breast_cancer: {X_train.shape[0]} linhas de treino | {X_test.shape[0]} de teste "
      f"| {X_train.shape[1]} atributos")
print(f"📊 baseline com todos os atributos: acurácia de teste = {baseline_accuracy:.4f}")

## Passo 3: evolucionar os subconjuntos

O fitness roda uma validação cruzada de 3 folds no treino e devolve a acurácia média. Cada avaliação treina 3 modelos, então o orçamento importa: 30 indivíduos por 25 gerações são 750 avaliações, algo em torno de 2250 treinos.

Um detalhe que evita resultado enganoso: subconjunto vazio recebe fitness 0, senão ele nunca seria descartado.


In [ ]:
folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)


def subset_fitness(genes: np.ndarray) -> float:
    """Mean CV accuracy of a LogisticRegression trained on the selected columns."""
    selected = np.flatnonzero(genes)
    if selected.size == 0:
        return 0.0
    scores = []
    for train_index, valid_index in folds.split(X_train, y_train):
        model = LogisticRegression(max_iter=5000)
        model.fit(X_train[train_index][:, selected], y_train[train_index])
        scores.append(model.score(X_train[valid_index][:, selected], y_train[valid_index]))
    return float(np.mean(scores))


ga_features = GeneticAlgorithm(
    fitness=subset_fitness,
    decode=lambda genes: genes,
    size=30, n_genes=X_train.shape[1],
    tournament=3, crossover_rate=0.9, mutation_rate=0.02, seed=RANDOM_STATE,
)

print("🧬 30 individuos, um gene por atributo, 25 geracoes")
best_mask, best_cv_accuracy = ga_features.run(generations=25, verbose=True)

In [ ]:
selected_names = data.feature_names[best_mask.astype(bool)]
selected_model = LogisticRegression(max_iter=5000).fit(X_train[:, best_mask.astype(bool)], y_train)
selected_accuracy = selected_model.score(X_test[:, best_mask.astype(bool)], y_test)

summary = pd.DataFrame(
    [
        {"modelo": "todos os atributos", "atributos": X_train.shape[1],
         "acuracia_cv": float("nan"), "acuracia_teste": baseline_accuracy},
        {"modelo": "subconjunto do GA", "atributos": int(best_mask.sum()),
         "acuracia_cv": best_cv_accuracy, "acuracia_teste": selected_accuracy},
    ]
).set_index("modelo")
display(summary.round(4))

print("🧬 atributos escolhidos:", ", ".join(selected_names))

### Como ler isso

O que interessa não é o subconjunto específico que saiu, e sim a troca: menos atributos para treinar e medir, com acurácia de teste na vizinhança da do modelo completo. Em dados médicos isso vale tempo e custo de exame.

No teste, o subconjunto pode ficar um pouco atrás ou à frente do baseline, e com 143 linhas de teste dois acertos de diferença já são ruído. O GA otimizou a acurácia de validação cruzada dentro do treino, que é otimista por construção, porque é ela que o fitness persegue.

Dois cuidados antes de tirar conclusão:

- o algoritmo genético é **estocástico**: troque a semente e o subconjunto muda. Para afirmar o que ele costuma escolher, rode várias sementes e olhe a frequência de cada atributo;
- uma única divisão de treino e teste não decide qual conjunto de atributos é melhor (a lição vale para tudo neste repositório: repita com sementes diferentes e reporte média e desvio).


## Resumo

O algoritmo genético troca derivada por população. Ele funciona onde a decisão é discreta ou combinatória e a função objetivo é uma caixa-preta que você só consegue avaliar, como uma validação cruzada. Em troca, cada geração custa tantas avaliações quantos forem os indivíduos, e o resultado é aleatório: a mesma receita com outra semente dá outro subconjunto.

As peças que você levaria para outro problema: uma representação (o cromossomo), um `decode` que traduz o cromossomo para o que o fitness lê, uma função de fitness, e três parâmetros que mandam no comportamento, tamanho da população, taxa de mutação e pressão da seleção.

**Próximo passo:** trocar o elitismo simples por seleção por ranking, ou usar dois pontos de corte no crossover, e comparar a curva de convergência. É aí que se aprende o que cada operador faz.

### Referências

**O algoritmo genético**

Holland, J. H. (1975). *Adaptation in Natural and Artificial Systems.* University of Michigan Press.

Goldberg, D. E. (1989). *Genetic Algorithms in Search, Optimization, and Machine Learning.* Addison-Wesley.

Whitley, D. (1994). *A genetic algorithm tutorial.* Statistics and Computing 4(2), 65-85.

**Seleção de atributos com computação evolutiva**

Xue, B., Zhang, M., Browne, W. N., & Yao, X. (2016). *A survey on evolutionary computation approaches to feature selection.* IEEE Transactions on Evolutionary Computation 20(4), 606-626.

```bibtex
@book{holland1975adaptation,
  title     = {Adaptation in Natural and Artificial Systems},
  author    = {Holland, John H.},
  year      = {1975},
  publisher = {University of Michigan Press}
}

@book{goldberg1989genetic,
  title     = {Genetic Algorithms in Search, Optimization, and Machine Learning},
  author    = {Goldberg, David E.},
  year      = {1989},
  publisher = {Addison-Wesley}
}

@article{whitley1994genetic,
  title   = {A genetic algorithm tutorial},
  author  = {Whitley, Darrell},
  journal = {Statistics and Computing},
  volume  = {4},
  number  = {2},
  pages   = {65--85},
  year    = {1994}
}

@article{xue2016survey,
  title   = {A survey on evolutionary computation approaches to feature selection},
  author  = {Xue, Bing and Zhang, Mengjie and Browne, Will N. and Yao, Xin},
  journal = {IEEE Transactions on Evolutionary Computation},
  volume  = {20},
  number  = {4},
  pages   = {606--626},
  year    = {2016}
}
```
